# RiskBricks Daily Data Refresh

Automated pipeline: **Bronze ingestion → Silver validation → Gold analytics recomputation**

> Scheduled to run daily after US market close (6 PM ET)

In [0]:
%pip install yfinance -q
dbutils.library.restartPython()

In [0]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

symbols = [r.symbol for r in spark.sql("SELECT DISTINCT symbol FROM riskbricks.gold.company_universe ORDER BY symbol").collect()]
end_date = datetime.now()
start_date = end_date - timedelta(days=10)  # 10 calendar days ≈ 5-7 trading days buffer

print(f"📥 Downloading {len(symbols)} symbols from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")

batch_size = 50
all_dfs = []
failed = []

for i in range(0, len(symbols), batch_size):
    batch = symbols[i:i+batch_size]
    batch_str = " ".join(batch)
    try:
        data = yf.download(batch_str, start=start_date.strftime('%Y-%m-%d'), 
                          end=end_date.strftime('%Y-%m-%d'), 
                          group_by='ticker', threads=True, progress=False)
        for sym in batch:
            try:
                if len(batch) == 1:
                    df = data.copy()
                else:
                    df = data[sym].copy()
                df = df.dropna(subset=['Close'])
                if not df.empty:
                    df = df.reset_index()
                    df['symbol'] = sym
                    if hasattr(df.columns, 'levels'):
                        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
                    all_dfs.append(df)
            except Exception:
                failed.append(sym)
    except Exception as e:
        failed.extend(batch)

combined = pd.concat(all_dfs, ignore_index=True)
combined = combined.rename(columns={'Date': 'date', 'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume'})
for col in ['Adj Close', 'adj_close', 'Adj_Close']:
    if col in combined.columns:
        combined = combined.drop(columns=[col])
combined['date'] = pd.to_datetime(combined['date']).dt.date

schema = StructType([
    StructField("date", DateType(), True), StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True), StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True), StructField("volume", LongType(), True),
    StructField("symbol", StringType(), True),
])
sdf = spark.createDataFrame(combined, schema=schema)
sdf = sdf.withColumn("ingestion_timestamp", F.current_timestamp()) \
         .withColumn("adj_close", F.col("close")) \
         .withColumn("dividends", F.lit(0.0)) \
         .withColumn("stock_splits", F.lit(0.0)) \
         .withColumn("capital_gains", F.lit(0.0)) \
         .withColumn("price", F.col("close"))

sdf.createOrReplaceTempView("new_stock_data")
spark.sql("""
    MERGE INTO riskbricks.bronze.stock_prices_bronze AS target
    USING new_stock_data AS source
    ON target.symbol = source.symbol AND target.date = source.date
    WHEN MATCHED THEN UPDATE SET
        target.open = source.open, target.high = source.high, target.low = source.low,
        target.close = source.close, target.volume = source.volume,
        target.adj_close = source.adj_close, target.ingestion_timestamp = source.ingestion_timestamp
    WHEN NOT MATCHED THEN INSERT (symbol, date, open, high, low, close, volume, adj_close, dividends, stock_splits, capital_gains, ingestion_timestamp, price)
    VALUES (source.symbol, source.date, source.open, source.high, source.low, source.close, source.volume, source.adj_close, source.dividends, source.stock_splits, source.capital_gains, source.ingestion_timestamp, source.price)
""")
result = spark.sql("SELECT MAX(date) AS latest, COUNT(*) AS total FROM riskbricks.bronze.stock_prices_bronze").collect()[0]
print(f"✅ Bronze: {len(all_dfs)} symbols ingested, latest={result.latest}, total={result.total:,} rows")
if failed:
    print(f"⚠️ {len(failed)} symbols failed (possibly delisted): {failed[:10]}")

In [0]:
from pyspark.sql import functions as F, Window

latest_silver = spark.sql("SELECT MAX(date) AS d FROM riskbricks.silver.stock_prices").collect()[0].d
bronze_new = spark.sql(f"""
    SELECT symbol, date, open, high, low, close, volume, adj_close
    FROM riskbricks.bronze.stock_prices_bronze WHERE date > '{latest_silver}'
""")
new_count = bronze_new.count()
if new_count == 0:
    print("ℹ️ No new data to process in silver layer")
else:
    w = Window.partitionBy("symbol").orderBy("date")
    silver_new = bronze_new \
        .withColumn("prev_close", F.lag("close").over(w)) \
        .withColumn("price_change_pct",
            F.when(F.col("prev_close").isNotNull() & (F.col("prev_close") != 0),
                   (F.col("close") - F.col("prev_close")) / F.col("prev_close") * 100).otherwise(0.0)) \
        .withColumn("is_anomaly", F.when(F.abs(F.col("price_change_pct")) > 20, True).otherwise(False)) \
        .withColumn("quality_score",
            F.when(F.col("close").isNotNull() & F.col("volume").isNotNull() & (F.col("volume") > 0) & (F.col("close") > 0), 1.0).otherwise(0.5)) \
        .withColumn("validated_at", F.current_timestamp()) \
        .drop("prev_close")
    silver_new.createOrReplaceTempView("silver_new_data")
    spark.sql("""
        MERGE INTO riskbricks.silver.stock_prices AS target
        USING silver_new_data AS source ON target.symbol = source.symbol AND target.date = source.date
        WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *
    """)
    result = spark.sql("SELECT MAX(date) AS latest, COUNT(*) AS total FROM riskbricks.silver.stock_prices").collect()[0]
    print(f"✅ Silver: {new_count:,} new rows, latest={result.latest}, total={result.total:,}")

In [0]:
spark.sql("""
    MERGE INTO riskbricks.gold.company_universe AS target
    USING (
        SELECT s.symbol, s.close AS latest_price, s.date AS price_date,
               ((s.close - LAG(s.close) OVER (PARTITION BY s.symbol ORDER BY s.date)) /
                NULLIF(LAG(s.close) OVER (PARTITION BY s.symbol ORDER BY s.date), 0)) * 100 AS price_change_1d_pct,
               STDDEV(s.price_change_pct) OVER (PARTITION BY s.symbol ORDER BY s.date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) / 100 AS vol_30d,
               CURRENT_TIMESTAMP() AS ts
        FROM riskbricks.silver.stock_prices s
        WHERE s.date = (SELECT MAX(date) FROM riskbricks.silver.stock_prices)
    ) AS source ON target.symbol = source.symbol
    WHEN MATCHED THEN UPDATE SET
        target.latest_price = source.latest_price,
        target.price_change_1d_pct = source.price_change_1d_pct,
        target.volatility_30d = COALESCE(source.vol_30d, target.volatility_30d),
        target.price_updated_at = source.ts
""")
cnt = spark.sql("SELECT COUNT(*) AS n FROM riskbricks.gold.company_universe WHERE price_updated_at > CURRENT_DATE()").collect()[0].n
print(f"✅ Company universe: {cnt} stocks updated with latest prices")

In [0]:
spark.sql("""
    MERGE INTO riskbricks.gold.portfolio_holdings AS h
    USING (SELECT c.symbol, c.latest_price, c.beta, c.volatility_30d FROM riskbricks.gold.company_universe c WHERE c.latest_price IS NOT NULL) AS src
    ON h.symbol = src.symbol
    WHEN MATCHED THEN UPDATE SET
        h.current_price = src.latest_price, h.value_usd = h.shares * src.latest_price,
        h.unrealized_gain_loss = (src.latest_price - h.avg_cost_per_share) * h.shares,
        h.unrealized_gain_loss_pct = CASE WHEN h.avg_cost_per_share > 0 THEN ((src.latest_price - h.avg_cost_per_share) / h.avg_cost_per_share) * 100 ELSE 0 END,
        h.beta = src.beta, h.volatility_30d = src.volatility_30d,
        h.last_price_update = CURRENT_TIMESTAMP(), h.as_of_date = CURRENT_DATE()
""")
spark.sql("""
    MERGE INTO riskbricks.gold.portfolio_holdings AS h
    USING (SELECT portfolio_id, symbol, value_usd / SUM(value_usd) OVER (PARTITION BY portfolio_id) AS new_weight
           FROM riskbricks.gold.portfolio_holdings WHERE value_usd IS NOT NULL AND value_usd > 0) AS w
    ON h.portfolio_id = w.portfolio_id AND h.symbol = w.symbol
    WHEN MATCHED THEN UPDATE SET h.weight = w.new_weight
""")
spark.sql("""
    MERGE INTO riskbricks.gold.portfolio_managers AS m
    USING (SELECT manager_id, SUM(value_usd) AS total_aum, COUNT(*) AS cnt,
                  SUM(weight * beta) AS wtd_beta, SUM(weight * volatility_30d) AS wtd_vol
           FROM riskbricks.gold.portfolio_holdings GROUP BY manager_id) AS h
    ON m.manager_id = h.manager_id
    WHEN MATCHED THEN UPDATE SET
        m.aum_usd = h.total_aum, m.num_holdings = h.cnt, m.portfolio_beta = h.wtd_beta,
        m.portfolio_volatility_pct = h.wtd_vol * 100, m.last_updated_at = CURRENT_TIMESTAMP()
""")
spark.sql("SELECT manager_name, ROUND(aum_usd/1e6,1) AS aum_m, num_holdings, ROUND(portfolio_beta,2) AS beta FROM riskbricks.gold.portfolio_managers").show()
print("✅ Holdings and manager AUM updated")

In [0]:
spark.sql("""
    INSERT OVERWRITE riskbricks.gold.portfolio_risk_metrics
    SELECT h.portfolio_id, h.manager_id, m.manager_name, m.risk_profile,
        SUM(h.value_usd) AS aum_usd,
        SUM(h.weight * h.volatility_30d) AS weighted_volatility_pct,
        SUM(h.weight * h.beta) AS portfolio_beta,
        AVG(h.volatility_30d / 100.0) AS avg_stock_volatility,
        COUNT(*) AS num_positions,
        SUM(h.value_usd) * (SUM(h.weight * h.volatility_30d) / 100.0 / SQRT(252)) * 1.645 AS var_1day_95_usd,
        SUM(h.value_usd) * (SUM(h.weight * h.volatility_30d) / 100.0 / SQRT(252)) * 1.645 * SQRT(10) AS var_10day_95_usd,
        CURRENT_TIMESTAMP() AS computed_at
    FROM riskbricks.gold.portfolio_holdings h
    JOIN riskbricks.gold.portfolio_managers m ON h.manager_id = m.manager_id
    WHERE h.value_usd IS NOT NULL AND h.value_usd > 0
    GROUP BY h.portfolio_id, h.manager_id, m.manager_name, m.risk_profile
""")
spark.sql("SELECT manager_name, ROUND(aum_usd/1e6,1) AS aum_m, ROUND(var_1day_95_usd,0) AS var_1d, ROUND(portfolio_beta,2) AS beta FROM riskbricks.gold.portfolio_risk_metrics").show()
print("✅ Risk metrics recomputed")

In [0]:
spark.sql("""
    INSERT OVERWRITE riskbricks.gold.stress_test_results
    SELECT r.portfolio_id, r.manager_id, r.manager_name, s.scenario_name, s.scenario_description,
        r.aum_usd, r.aum_usd * s.shock_pct / 100.0 * r.portfolio_beta AS total_impact_usd,
        s.shock_pct * r.portfolio_beta AS impact_pct, CURRENT_TIMESTAMP() AS computed_at
    FROM riskbricks.gold.portfolio_risk_metrics r
    CROSS JOIN (
        SELECT 'Market Crash (-20%)' AS scenario_name, 'S&P 500 drops 20%' AS scenario_description, -20.0 AS shock_pct
        UNION ALL SELECT 'Rate Hike (+200bp)', 'Fed raises rates 200bp, equity drop ~8%', -8.0
        UNION ALL SELECT 'Recession', 'GDP contracts, broad market selloff ~15%', -15.0
        UNION ALL SELECT 'Bull Rally (+15%)', 'Strong market rally +15%', 15.0
    ) s
""")
spark.sql("""
    INSERT OVERWRITE riskbricks.gold.sector_exposures
    SELECT h.portfolio_id, h.manager_id, m.manager_name, h.sector,
        SUM(h.weight) * 100 AS sector_weight_pct, CURRENT_TIMESTAMP() AS computed_at
    FROM riskbricks.gold.portfolio_holdings h
    JOIN riskbricks.gold.portfolio_managers m ON h.manager_id = m.manager_id
    WHERE h.value_usd IS NOT NULL AND h.value_usd > 0
    GROUP BY h.portfolio_id, h.manager_id, m.manager_name, h.sector
""")
print("✅ Stress tests: " + str(spark.sql("SELECT COUNT(*) FROM riskbricks.gold.stress_test_results").collect()[0][0]) + " scenarios")
print("✅ Sector exposures: " + str(spark.sql("SELECT COUNT(*) FROM riskbricks.gold.sector_exposures").collect()[0][0]) + " rows")

In [0]:
print("=" * 60)
print("🎉 DAILY REFRESH COMPLETE")
print("=" * 60)
for table, query in [
    ("bronze.stock_prices_bronze", "SELECT MAX(date) AS d, COUNT(*) AS n FROM riskbricks.bronze.stock_prices_bronze"),
    ("silver.stock_prices", "SELECT MAX(date) AS d, COUNT(*) AS n FROM riskbricks.silver.stock_prices"),
    ("gold.company_universe", "SELECT MAX(price_updated_at) AS d, COUNT(*) AS n FROM riskbricks.gold.company_universe"),
    ("gold.portfolio_risk_metrics", "SELECT MAX(computed_at) AS d, COUNT(*) AS n FROM riskbricks.gold.portfolio_risk_metrics"),
    ("gold.stress_test_results", "SELECT MAX(computed_at) AS d, COUNT(*) AS n FROM riskbricks.gold.stress_test_results"),
    ("gold.sector_exposures", "SELECT MAX(computed_at) AS d, COUNT(*) AS n FROM riskbricks.gold.sector_exposures"),
]:
    r = spark.sql(query).collect()[0]
    print(f"  ✅ {table}: latest={r.d}, rows={r.n}")
print("=" * 60)